# 🧠 Enhanced Financial Analyst Agent: Two-Stage AI Integration
## Part 2: OpenAI Responses API Integration with Mathematical Pre-filtering

**Learning Objectives:**
- Integrate the OpenAI Responses API for sophisticated financial analysis
- Implement the two-stage approach: Mathematical filtering → AI analysis
- Compare aligned (pre-filtered) vs naive (raw) AI decision-making
- Understand how mathematical pre-filtering can improve AI reliability

**What You'll Build:**
A complete two-stage financial AI system that combines mathematical document scoring with advanced AI analysis, illustrating how pre-filtering can improve on naive approaches. (This notebook is a single-run demonstration per client profile — the rigorous, repeated-trial evaluation of the comparison is Part 3's job.)

**Prerequisites:**
- Completion of Part 1: Mathematical Alignment Framework
- OpenAI API key for Responses API access (optional — simulation mode works without one)
- Understanding of AI assistant design principles


## 🔄 Recap: Two-Stage Architecture

### Stage 1: Mathematical Document Scoring ✅ (From Part 1)
- Deterministic, auditable document ranking
- Client-specific personalization
- Risk and credibility assessment
- Quality-assured information filtering

### Stage 2: AI Analysis 🚀 (This Notebook)
- **Aligned Approach**: AI analyzes mathematically pre-selected documents
- **Naive Approach**: AI chooses and analyzes documents independently
- **Comparative Analysis**: Measure the impact of mathematical pre-filtering

### Key Innovation
By separating document selection (mathematical) from analysis (AI), we:
- Prevent AI from being misled by low-quality sources
- Ensure consistent quality standards
- Create auditable decision trails
- Enable systematic personalization

## 📚 Setup and Dependencies

In [ ]:
import numpy as np
import openai
import os
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from datetime import datetime
import json

# Model is configurable; gpt-5.5 is OpenAI's current recommended model (June 2026)
MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.5")

print("🤖 Financial AI Integration Framework Loading...")
print("📋 Checking OpenAI API availability...")

# Check for OpenAI API key
api_key = None

# Method 1: Environment variable
if 'OPENAI_API_KEY' in os.environ:
    api_key = os.environ['OPENAI_API_KEY']
    print("✅ Found OpenAI API key in environment variables")
else:
    # Method 2: Google Colab userdata (no-op outside Colab)
    try:
        from google.colab import userdata
        api_key = userdata.get('OPENAI_API_KEY')
        print("✅ Found OpenAI API key in Colab userdata")
    except ImportError:
        pass
    except Exception:
        print("⚠️ Running in Colab but no API key found in userdata")

if api_key and api_key.startswith('sk-'):
    client = openai.OpenAI(api_key=api_key)
    print("✅ OpenAI client initialized successfully")
    OPENAI_AVAILABLE = True
else:
    print("❌ No valid OpenAI API key found")
    print("💡 Please set your API key: export OPENAI_API_KEY='sk-your-key-here'")
    print("📋 Or in Colab: Add 'OPENAI_API_KEY' to Secrets")
    OPENAI_AVAILABLE = False
    client = None

print(f"\n🔧 Setup Status:")
print(f"   OpenAI API: {'✅ Available' if OPENAI_AVAILABLE else '❌ Not Available'}")
print(f"   Model: {MODEL}")
print(f"   Mode: {'Full Demo' if OPENAI_AVAILABLE else 'Simulation Only'}")


## 📊 Import Core Components from Part 1

We need the mathematical scoring framework and client profiles from Part 1.

In [ ]:
# Core classes and functions from Part 1
# Copy these if Part 1 isn't available in your environment

@dataclass
class ClientProfile:
    """Client profile for personalized document scoring"""
    risk_tolerance: float  # 0-1 scale
    investment_horizon: int  # years
    portfolio_size: float  # in millions USD
    sectors_of_interest: List[str]
    current_positions: Dict[str, float]  # symbol -> weight
    
    def to_feature_vector(self) -> np.ndarray:
        return np.array([
            self.risk_tolerance,
            min(self.investment_horizon / 20, 1.0),
            min(self.portfolio_size / 100, 1.0),
            len(self.sectors_of_interest) / 10,
        ])
    
    def get_sector_preferences(self) -> Dict[str, float]:
        if not self.sectors_of_interest:
            return {}
        weight = 1.0 / len(self.sectors_of_interest)
        return {sector: weight for sector in self.sectors_of_interest}

def compute_alignment_score(R_d, S_d, X_d, w, n_d, delta_t_d, tau, velocity, credibility,
                          alpha=1.0, beta=1.0, lambda_=0.5, gamma_1=0.6, gamma_2=0.4,
                          risk_penalty=0.1, confidence_boost=0.2):
    """Enhanced alignment scoring with risk assessment"""
    # Core alignment score components
    E_d = (1 / np.sqrt(n_d + 1)) * np.exp(-delta_t_d / tau)
    Q_d = gamma_1 * velocity + gamma_2 * credibility
    base_score = alpha * R_d + beta * S_d + np.dot(w, X_d) + lambda_ * E_d * Q_d

    # Risk assessment
    risk_score = (1 - credibility) * velocity * risk_penalty
    aligned_score = base_score - risk_score

    # Confidence interval
    confidence = min(0.95, credibility + (n_d / 10) * confidence_boost)
    uncertainty = (1 - confidence) * 0.1 * base_score

    return aligned_score, (aligned_score - uncertainty, aligned_score + uncertainty), risk_score

def compute_client_document_match(doc, client):
    """Compute client-document alignment score"""
    # Sector relevance
    doc_sectors = doc.get('sectors', [])
    client_sector_prefs = client.get_sector_preferences()
    sector_match = sum(client_sector_prefs.get(sector, 0) for sector in doc_sectors)
    
    # Risk alignment
    doc_risk = doc.get('risk_level', 0.5)
    risk_match = 1 - abs(doc_risk - client.risk_tolerance)
    
    # Position relevance
    symbols_mentioned = doc.get('symbols', [])
    position_relevance = sum(client.current_positions.get(symbol, 0) for symbol in symbols_mentioned)
    
    # Portfolio impact factor
    portfolio_factor = min(client.portfolio_size / 10, 1.0)
    
    # Time horizon alignment
    doc_time_relevance = 1.0 - (doc.get('delta_t_d', 0) / 1440)
    horizon_match = 1.0 - abs(doc_time_relevance - min(client.investment_horizon / 10, 1.0))
    
    # Weighted combination
    weights = np.array([0.3, 0.25, 0.2, 0.15, 0.1])
    factors = np.array([sector_match, risk_match, min(position_relevance, 1.0), portfolio_factor, horizon_match])
    
    return np.dot(weights, factors)

# Sample documents and clients (simplified from Part 1)
documents = [
    {
        "title": "Microsoft Q3 Earnings Beat: Cloud Revenue Surges 28%",
        "body": "Microsoft reported Q3 earnings of $2.45 per share, beating analyst estimates of $2.23. Azure cloud services drove growth with 28% YoY revenue increase. Enterprise software adoption accelerated as hybrid work trends continue. Management raised full-year guidance citing strong enterprise demand.",
        "source": "Financial Times",
        "R_d": 0.92, "S_d": 0.89, "X_d": np.array([0.8, 0.3, 0.9, 0.7]),
        "n_d": 12, "delta_t_d": 180, "velocity": 0.08, "credibility": 0.94,
        "sectors": ["Technology", "Cloud Computing"], "symbols": ["MSFT"], "risk_level": 0.3
    },
    {
        "title": "EU Antitrust Probe: Microsoft Faces €2.4B Fine Over Teams Bundling",
        "body": "European Commission investigators recommend substantial fines against Microsoft for allegedly anti-competitive bundling of Teams with Office 365. Sources suggest penalties could reach €2.4 billion. Microsoft disputes claims, citing competitor integrations. Regulatory uncertainty may impact EU expansion plans.",
        "source": "Reuters Legal",
        "R_d": 0.75, "S_d": 0.71, "X_d": np.array([0.6, 0.7, 0.4, 0.8]),
        "n_d": 8, "delta_t_d": 120, "velocity": 0.18, "credibility": 0.87,
        "sectors": ["Technology", "Legal/Regulatory"], "symbols": ["MSFT"], "risk_level": 0.7
    },
    {
        "title": "EXCLUSIVE: Microsoft May Exit Surface Hardware by 2025 - Sources",
        "body": "Three industry sources familiar with Microsoft's internal strategy discussions suggest the company is considering exiting the Surface hardware business by 2025. Reasons cited include supply chain challenges and focus on software/cloud services. No official confirmation from Microsoft. Stock analysts divided on impact.",
        "source": "TechCrunch (Unnamed Sources)",
        "R_d": 0.58, "S_d": 0.55, "X_d": np.array([0.4, 0.8, 0.3, 0.4]),
        "n_d": 3, "delta_t_d": 45, "velocity": 0.25, "credibility": 0.42,
        "sectors": ["Technology", "Hardware"], "symbols": ["MSFT"], "risk_level": 0.8
    },
    {
        "title": "Microsoft Announces $10B AI Research Investment Over 3 Years",
        "body": "Microsoft unveiled a comprehensive $10 billion investment in AI research and development over the next three years. The initiative includes partnerships with leading universities, expansion of Azure AI services, and development of next-generation copilot technologies. CEO Satya Nadella emphasized the strategic importance of AI leadership.",
        "source": "Microsoft Press Release",
        "R_d": 0.84, "S_d": 0.82, "X_d": np.array([0.7, 0.4, 0.95, 0.9]),
        "n_d": 15, "delta_t_d": 360, "velocity": 0.12, "credibility": 0.98,
        "sectors": ["Technology", "Artificial Intelligence"], "symbols": ["MSFT"], "risk_level": 0.2
    }
]

# Sample client profiles
conservative_client = ClientProfile(
    risk_tolerance=0.3, investment_horizon=10, portfolio_size=2.5,
    sectors_of_interest=["Technology", "Healthcare"],
    current_positions={"MSFT": 0.15, "AAPL": 0.12, "JNJ": 0.20}
)

aggressive_client = ClientProfile(
    risk_tolerance=0.8, investment_horizon=5, portfolio_size=0.8,
    sectors_of_interest=["Technology", "Artificial Intelligence", "Cloud Computing"],
    current_positions={"MSFT": 0.25, "NVDA": 0.30, "GOOGL": 0.20}
)

print("📊 Core components loaded successfully!")
print(f"📋 Documents available: {len(documents)}")
print(f"👥 Client profiles: Conservative & Aggressive")
print(f"🔧 Mathematical scoring functions: Ready")

## 🧮 Stage 1: Mathematical Document Ranking

First, let's implement the mathematical document scoring that will feed into our AI analysis.

In [ ]:
def perform_mathematical_document_ranking(documents, client_profile, query_context="Microsoft investment analysis"):
    """
    Stage 1: Mathematical document scoring and ranking
    This creates the quality-filtered input for Stage 2 AI analysis
    """
    print(f"🔢 STAGE 1: MATHEMATICAL DOCUMENT RANKING")
    print(f"📋 Context: {query_context}")
    print(f"👤 Client: {client_profile.risk_tolerance:.1f} risk tolerance")
    print("-" * 60)

    base_weights = client_profile.to_feature_vector()
    scored_documents = []

    for i, doc in enumerate(documents):
        # Compute client-document alignment
        client_match = compute_client_document_match(doc, client_profile)
        enhanced_X_d = np.concatenate([doc['X_d'], [client_match]])
        enhanced_weights = np.concatenate([base_weights, [1.0]])

        # Apply alignment scoring equation
        score, confidence_interval, risk_score = compute_alignment_score(
            doc['R_d'], doc['S_d'], enhanced_X_d, enhanced_weights,
            doc['n_d'], doc['delta_t_d'], tau=300,
            velocity=doc['velocity'], credibility=doc['credibility']
        )

        print(f"📄 {i+1}. {doc['title'][:45]}...")
        print(f"    Score: {score:.4f} | Credibility: {doc['credibility']:.2f} | Risk: {doc['risk_level']:.1f}")

        doc_scored = doc.copy()
        doc_scored.update({
            'alignment_score': score,
            'confidence_interval': confidence_interval,
            'risk_score': risk_score,
            'client_match': client_match
        })
        scored_documents.append(doc_scored)

    # Rank by alignment score
    ranked_documents = sorted(scored_documents, key=lambda x: x['alignment_score'], reverse=True)

    print(f"\n🏆 MATHEMATICAL RANKING RESULTS:")
    for i, doc in enumerate(ranked_documents):
        print(f"   {i+1}. {doc['title'][:50]}... (Score: {doc['alignment_score']:.3f})")

    top_document = ranked_documents[0]
    print(f"\n✅ Top-ranked document for AI analysis:")
    print(f"   📰 {top_document['title']}")
    print(f"   📊 Score: {top_document['alignment_score']:.4f}")
    print(f"   🔍 Credibility: {top_document['credibility']:.3f}")
    print(f"   ⚖️ Client Match: {top_document['client_match']:.3f}")

    return ranked_documents, top_document

# Demonstrate mathematical ranking for both client types
print("🔬 MATHEMATICAL DOCUMENT RANKING DEMONSTRATION\n")

print("🔷 CONSERVATIVE CLIENT RANKING:")
conservative_ranked, conservative_top = perform_mathematical_document_ranking(
    documents, conservative_client, "Conservative Microsoft analysis"
)

print("\n" + "🔄" * 20 + "\n")

print("🔶 AGGRESSIVE CLIENT RANKING:")
aggressive_ranked, aggressive_top = perform_mathematical_document_ranking(
    documents, aggressive_client, "Aggressive Microsoft analysis"
)

## 🤖 Stage 2: OpenAI Responses API Integration

Now we'll create specialized financial analyst roles that can operate in two modes:
1. **Aligned Mode**: Analyzes mathematically pre-selected documents
2. **Naive Mode**: Chooses and analyzes documents independently

**How the API works**: each analysis is one synchronous call to
`client.responses.create(model=..., instructions=..., input=[...])` — the
`instructions` parameter plays the role a system prompt does, and the reply
text comes back directly on the response object. No server-side assistant
objects, no threads, no run-status polling. (Multi-turn state exists via
`previous_response_id`, but each analysis here is single-turn, so we don't
need it.)

> **A teaching note on API churn**: an earlier version of this notebook used
> the OpenAI *Assistants* API (assistant + thread + run-polling objects).
> OpenAI deprecated it in August 2025 and removes the endpoints permanently
> on **August 26, 2026**. Porting this notebook took rewriting ~150 lines of
> plumbing into one API call. Vendor API churn is a real, recurring
> maintenance cost in production AI systems — one more reason the *logic* of
> your system (Stage 1's scoring math) should never be entangled with the
> *plumbing* (Stage 2's API client).


In [ ]:
ANALYST_INSTRUCTIONS = """You are an expert financial analyst assistant specializing in investment analysis and market research.

Your expertise includes:
• **Investment Analysis**: Evaluating financial news and market developments
• **Risk Assessment**: Identifying potential downsides and volatility factors
• **Portfolio Impact**: Understanding how news affects investment portfolios
• **Client Alignment**: Tailoring advice to specific client risk profiles
• **Source Evaluation**: Assessing credibility and reliability of information

**Analysis Framework:**
1. Source credibility assessment
2. Information relevance and timeliness
3. Market impact analysis
4. Risk-return implications
5. Client-specific recommendations

Provide thorough, professional financial analysis with clear reasoning and appropriate confidence levels.
Always consider both upside potential and downside risks.
Tailor recommendations to the specific client profile provided.

You have no knowledge of previous analyses or other approaches. Each request is independent."""


class FinancialAnalystAssistant:
    """
    Stage 2: OpenAI Responses API integration for financial analysis.
    Supports both aligned (pre-filtered) and naive (independent) analysis modes.

    Each analysis is a single, stateless `responses.create` call — the
    instructions string above defines the analyst persona, and the document
    material travels in the user message.
    """

    def __init__(self, api_key: str = None, assistant_name: str = "Financial Analyst"):
        self.assistant_name = assistant_name

        if OPENAI_AVAILABLE and client:
            self.client = client
            print(f"✅ {self.assistant_name} ready (model: {MODEL})")
        else:
            self.client = None
            print(f"⚠️ OpenAI not available - {assistant_name} will run in simulation mode")

    def analyze_with_mathematical_alignment(self, top_document, client_profile):
        """
        ALIGNED APPROACH: AI analyzes the mathematically pre-selected best document
        The AI receives the document that passed mathematical quality filters
        """
        if not self.client:
            return self._simulate_aligned_analysis(top_document, client_profile)

        prompt = f"""🧮 ALIGNED FINANCIAL ANALYSIS

**Document Selected by Mathematical Alignment Scoring:**

📰 **Title:** {top_document['title']}
📊 **Source:** {top_document['source']}
⭐ **Mathematical Alignment Score:** {top_document.get('alignment_score', 'N/A'):.4f}
🔍 **Source Credibility Rating:** {top_document['credibility']:.3f}/1.0
⚠️ **Risk Assessment:** {top_document.get('risk_score', 'N/A'):.4f}
🎯 **Client Match Score:** {top_document.get('client_match', 'N/A'):.3f}/1.0
📊 **Analyst Coverage:** {top_document['n_d']} professional sources
⏰ **Document Age:** {top_document['delta_t_d']/60:.1f} hours

**Client Profile:**
• Risk Tolerance: {client_profile.risk_tolerance:.1f}/1.0 ({'Conservative' if client_profile.risk_tolerance < 0.4 else 'Moderate' if client_profile.risk_tolerance < 0.7 else 'Aggressive'})
• Investment Horizon: {client_profile.investment_horizon} years
• Portfolio Size: ${client_profile.portfolio_size:.1f}M
• Current MSFT Position: {client_profile.current_positions.get('MSFT', 0)*100:.1f}% of portfolio
• Sector Interests: {', '.join(client_profile.sectors_of_interest)}

**Document Content:**
{top_document['body']}

**Analysis Request:**
This document was pre-selected using mathematical alignment scoring to ensure optimal relevance, credibility, and client fit.
Please provide your comprehensive investment analysis and recommendations based on this high-quality, pre-filtered information.

**Please structure your analysis as:**
1. **Executive Summary** - Key takeaways and recommendation
2. **Market Impact Analysis** - How this affects Microsoft's prospects
3. **Risk Assessment** - Potential downsides and uncertainties
4. **Client-Specific Recommendations** - Tailored to this client's profile
5. **Confidence Level** - Your confidence in this analysis (1-10)
        """

        return self._get_model_response(prompt)

    def analyze_without_mathematical_guidance(self, all_documents, client_profile):
        """
        NAIVE APPROACH: AI chooses document itself and analyzes (no mathematical pre-filtering)
        Raw documents only - no credibility scores, alignment hints, or mathematical assistance
        """
        if not self.client:
            return self._simulate_naive_analysis(all_documents, client_profile)

        # Present ONLY raw document information - no scoring hints
        documents_summary = ""
        for i, doc in enumerate(all_documents):
            # Calculate age in a natural format without exposing internal variables
            hours_old = doc['delta_t_d'] / 60
            if hours_old < 1:
                age_str = f"{doc['delta_t_d']} minutes ago"
            elif hours_old < 24:
                age_str = f"{hours_old:.1f} hours ago"
            else:
                age_str = f"{hours_old/24:.1f} days ago"

            documents_summary += f"""
📄 **Document {i+1}:** {doc['title']}
📰 **Source:** {doc['source']}
⏰ **Published:** {age_str}
📖 **Content:** {doc['body']}

"""

        prompt = f"""🕐 INDEPENDENT FINANCIAL ANALYSIS

**Your Task:**
1. **Review** all available documents below
2. **Choose** the BEST document for analyzing Microsoft's investment prospects
3. **Provide** your investment analysis based on your chosen document

**Client Profile:**
• Risk Tolerance: {client_profile.risk_tolerance:.1f}/1.0 ({'Conservative' if client_profile.risk_tolerance < 0.4 else 'Moderate' if client_profile.risk_tolerance < 0.7 else 'Aggressive'})
• Investment Horizon: {client_profile.investment_horizon} years
• Portfolio Size: ${client_profile.portfolio_size:.1f}M
• Current MSFT Position: {client_profile.current_positions.get('MSFT', 0)*100:.1f}% of portfolio
• Sector Interests: {', '.join(client_profile.sectors_of_interest)}

**Available Documents:**
{documents_summary}

**Instructions:**
First, **explain which document you chose and why**, based solely on the information provided above.
Consider factors like source credibility, relevance, and timeliness using your professional judgment.

Then provide your investment analysis structured as:
1. **Document Selection Rationale** - Why you chose this document
2. **Executive Summary** - Key takeaways and recommendation
3. **Market Impact Analysis** - How this affects Microsoft's prospects
4. **Risk Assessment** - Potential downsides and uncertainties
5. **Client-Specific Recommendations** - Tailored to this client's profile
6. **Confidence Level** - Your confidence in this analysis (1-10)
        """

        response = self._get_model_response(prompt)

        # Try to extract which document was chosen
        chosen_doc_index = self._extract_chosen_document(response, all_documents)
        chosen_document = all_documents[chosen_doc_index] if chosen_doc_index is not None else all_documents[0]

        return response, chosen_document

    def _extract_chosen_document(self, response, all_documents):
        """Extract which document the AI chose from its response"""
        import re

        response_lower = response.lower()

        # Method 1: Look for "Document X" pattern
        doc_match = re.search(r'document (\d+)', response_lower)
        if doc_match:
            doc_num = int(doc_match.group(1))
            if 1 <= doc_num <= len(all_documents):
                return doc_num - 1  # Convert to 0-based index

        # Method 2: Look for document titles
        for i, doc in enumerate(all_documents):
            title_words = doc['title'].lower().split()[:4]  # First 4 words
            if all(word in response_lower for word in title_words):
                return i

        # Method 3: Look for source mentions
        for i, doc in enumerate(all_documents):
            if doc['source'].lower() in response_lower:
                return i

        return 0  # Default to first document

    def _get_model_response(self, prompt):
        """One synchronous Responses API call - no threads, no polling"""
        try:
            response = self.client.responses.create(
                model=MODEL,
                instructions=ANALYST_INSTRUCTIONS,
                input=[{"role": "user", "content": prompt}],
            )

            # Robust output extraction (response.output_text is the SDK shortcut)
            text = getattr(response, "output_text", None)
            if text:
                return text
            parts = []
            for item in response.output:
                if item.type == "message":
                    for content in item.content:
                        if content.type == "output_text":
                            parts.append(content.text)
            return "\n".join(parts) if parts else "❌ Empty model response"

        except Exception as e:
            return f"❌ Error during AI analysis: {e}"

    def _simulate_aligned_analysis(self, top_document, client_profile):
        """Simulation mode for aligned analysis when OpenAI isn't available"""
        return f"""🔬 SIMULATED ALIGNED ANALYSIS

**Document:** {top_document['title']}
**Alignment Score:** {top_document.get('alignment_score', 0):.4f}
**Credibility:** {top_document['credibility']:.3f}

**Simulated Analysis:**
Based on the mathematically pre-selected document with high credibility ({top_document['credibility']:.1%}) and strong alignment score, this appears to be a {"positive" if top_document.get('alignment_score', 0) > 1.5 else "mixed"} development for Microsoft.

For a client with {client_profile.risk_tolerance:.1f} risk tolerance, this news would suggest {"maintaining current position" if client_profile.risk_tolerance < 0.5 else "considering position adjustments"}.

💡 **Note:** This is a simulation. Run with OpenAI API key for full analysis.
        """

    def _simulate_naive_analysis(self, all_documents, client_profile):
        """Simulation mode for naive analysis when OpenAI isn't available"""
        # Simple heuristic: choose most recent document
        chosen_doc = min(all_documents, key=lambda x: x['delta_t_d'])

        analysis = f"""🔬 SIMULATED NAIVE ANALYSIS

**Document Selected:** {chosen_doc['title']}
**Selection Rationale:** Most recent information (simple heuristic)
**Credibility:** {chosen_doc['credibility']:.3f} (unknown to naive approach)

**Simulated Analysis:**
Selected the most recent document without mathematical guidance. This may not be the optimal choice for analysis quality or client alignment.

💡 **Note:** This is a simulation. Run with OpenAI API key for full analysis.
        """

        return analysis, chosen_doc

# Create analyst instances
if OPENAI_AVAILABLE:
    print("🤖 Creating Financial Analyst roles...")
    aligned_assistant = FinancialAnalystAssistant(assistant_name="Aligned Financial Analyst")
    naive_assistant = FinancialAnalystAssistant(assistant_name="Independent Financial Analyst")
    print("✅ Analysts ready for analysis")
else:
    print("🔬 Creating simulation analysts...")
    aligned_assistant = FinancialAnalystAssistant(assistant_name="Aligned Financial Analyst (Simulation)")
    naive_assistant = FinancialAnalystAssistant(assistant_name="Independent Financial Analyst (Simulation)")
    print("✅ Simulation mode ready")


## 🔬 Comparative Analysis: Aligned vs Naive Approaches

Now let's run both approaches side-by-side to see the impact of mathematical pre-filtering.

In [ ]:
def run_comparative_analysis(documents, client_profile, query="Microsoft investment analysis"):
    """
    Run complete comparative analysis: Aligned (2-stage) vs Naive (AI chooses) approaches
    """
    print(f"🚀 COMPARATIVE ANALYSIS: ALIGNED vs NAIVE APPROACHES")
    print(f"📋 Query: {query}")
    print(f"👤 Client: {client_profile.risk_tolerance:.1f} risk tolerance, {client_profile.investment_horizon}yr horizon")
    print("="*80)

    # STAGE 1: Mathematical Document Scoring (Aligned Approach Only)
    print(f"\n🔢 STAGE 1: MATHEMATICAL DOCUMENT SCORING (Aligned Approach)")
    print("-"*60)

    ranked_docs, top_aligned_doc = perform_mathematical_document_ranking(
        documents, client_profile, query
    )

    # STAGE 2: AI Analysis Comparison
    print(f"\n🤖 STAGE 2: AI ANALYSIS COMPARISON")
    print("-"*60)

    print("📊 Running ALIGNED analysis (AI analyzes mathematically pre-selected document)...")
    aligned_analysis = aligned_assistant.analyze_with_mathematical_alignment(top_aligned_doc, client_profile)

    print("📊 Running NAIVE analysis (AI chooses from raw documents independently)...")
    if OPENAI_AVAILABLE:
        naive_analysis, ai_chosen_doc = naive_assistant.analyze_without_mathematical_guidance(documents, client_profile)
    else:
        naive_analysis, ai_chosen_doc = naive_assistant._simulate_naive_analysis(documents, client_profile)

    # Display Results
    print(f"\n🎯 ALIGNED APPROACH RESULTS:")
    print("="*70)
    print(f"📰 Document Used: {top_aligned_doc['title']}")
    print(f"📊 Selection Method: Mathematical alignment scoring (Stage 1)")
    print(f"⭐ Alignment Score: {top_aligned_doc.get('alignment_score', 0):.4f}")
    print(f"🔍 Credibility: {top_aligned_doc['credibility']:.3f}")
    print(f"⚠️ Risk Level: {top_aligned_doc['risk_level']:.1f}")
    print(f"\n📝 Analysis:")
    print(aligned_analysis)

    print(f"\n⚡ NAIVE APPROACH RESULTS:")
    print("="*70)
    print(f"📰 Document Used: {ai_chosen_doc['title'] if ai_chosen_doc else 'Unknown'}")
    print(f"📊 Selection Method: AI's independent judgment (no mathematical guidance)")
    if ai_chosen_doc:
        print(f"🔍 Actual Credibility: {ai_chosen_doc['credibility']:.3f} (hidden from AI)")
        print(f"⚠️ Actual Risk Level: {ai_chosen_doc['risk_level']:.1f} (hidden from AI)")
    print(f"\n📝 Analysis:")
    print(naive_analysis)

    # Comparative Insights
    print(f"\n📈 COMPARATIVE INSIGHTS:")
    print("="*70)

    if ai_chosen_doc:
        print(f"📊 Document Selection Comparison:")
        print(f"   🧮 Mathematical Choice: {top_aligned_doc['title'][:55]}...")
        print(f"   🤖 AI's Independent Choice: {ai_chosen_doc['title'][:55]}...")
        
        same_document = top_aligned_doc['title'] == ai_chosen_doc['title']
        print(f"   📋 Same Document Selected: {'✅ YES' if same_document else '❌ NO'}")

        if not same_document:
            cred_diff = top_aligned_doc['credibility'] - ai_chosen_doc['credibility']
            risk_diff = ai_chosen_doc['risk_level'] - top_aligned_doc['risk_level']  # Higher is worse
            relevance_diff = top_aligned_doc['R_d'] - ai_chosen_doc['R_d']

            print(f"\n📊 Quality Metrics Comparison (hidden from Naive AI):")
            print(f"   🔍 Credibility: Math={top_aligned_doc['credibility']:.3f} vs AI={ai_chosen_doc['credibility']:.3f} (Δ={cred_diff:+.3f})")
            print(f"   ⚠️ Risk Level: Math={top_aligned_doc['risk_level']:.1f} vs AI={ai_chosen_doc['risk_level']:.1f} (Δ={-risk_diff:+.1f})")
            print(f"   📈 Relevance: Math={top_aligned_doc['R_d']:.3f} vs AI={ai_chosen_doc['R_d']:.3f} (Δ={relevance_diff:+.3f})")
            print(f"   📊 Analyst Coverage: Math={top_aligned_doc['n_d']} vs AI={ai_chosen_doc['n_d']} sources")

            print(f"\n🎯 Mathematical Pre-filtering Benefits:")
            if cred_diff > 0:
                print(f"   ✅ Selected higher credibility source (+{cred_diff:.3f})")
            if risk_diff > 0:
                print(f"   ✅ Selected lower risk content (-{risk_diff:.1f})")
            if relevance_diff > 0:
                print(f"   ✅ Selected more relevant information (+{relevance_diff:.3f})")
            if top_aligned_doc['n_d'] > ai_chosen_doc['n_d']:
                print(f"   ✅ Selected better verified source (+{top_aligned_doc['n_d'] - ai_chosen_doc['n_d']} sources)")

            if cred_diff <= 0 and risk_diff <= 0 and relevance_diff <= 0:
                print(f"   ⚠️ In this case, AI's intuitive choice performed well despite lack of mathematical guidance")
                print(f"   💡 However, mathematical approach provides consistent, auditable quality assurance")
        else:
            print(f"\n🎯 Interesting Convergence:")
            print(f"   • Both approaches selected the same document")
            print(f"   • This suggests the mathematical scoring aligned with AI intuition")
            print(f"   • However, mathematical approach provides consistent, repeatable selection")
            print(f"   • AI choices may vary between runs without mathematical guidance")

    print(f"\n🔑 KEY INSIGHTS FROM THIS COMPARISON:")
    print(f"   🧮 Aligned Approach: Systematic quality control + AI analysis")
    print(f"   🤖 Naive Approach: AI makes all decisions without mathematical foundation")
    print(f"   📊 Mathematical scoring ensures consistent information quality standards")
    print(f"   ⚖️ Two-stage approach separates quality control from content analysis")
    print(f"   🎯 Client-specific alignment is systematic rather than hoping AI notices")

    return {
        'aligned': {'document': top_aligned_doc, 'analysis': aligned_analysis},
        'naive': {'document': ai_chosen_doc, 'analysis': naive_analysis},
        'ranked_documents': ranked_docs
    }

# Run comparative analysis for conservative client
print("🔬 COMPREHENSIVE COMPARATIVE ANALYSIS\n")

conservative_results = run_comparative_analysis(
    documents, conservative_client, "Microsoft investment analysis for conservative portfolio"
)

In [ ]:
# Run comparative analysis for aggressive client
print("\n" + "🔄"*40 + "\n")
print("🔶 AGGRESSIVE CLIENT COMPARATIVE ANALYSIS\n")

aggressive_results = run_comparative_analysis(
    documents, aggressive_client, "Microsoft investment analysis for aggressive portfolio"
)

## 📊 Analysis Results Summary

Let's summarize the key findings from our comparative analysis.

In [ ]:
def summarize_comparative_results(conservative_results, aggressive_results):
    """Provide comprehensive summary of comparative analysis results"""
    
    print("📊 COMPREHENSIVE RESULTS SUMMARY")
    print("="*80)
    
    # Document Selection Analysis
    print("\n🎯 DOCUMENT SELECTION PATTERNS:")
    print("-"*50)
    
    cons_aligned_doc = conservative_results['aligned']['document']
    cons_naive_doc = conservative_results['naive']['document']
    agg_aligned_doc = aggressive_results['aligned']['document']
    agg_naive_doc = aggressive_results['naive']['document']
    
    print(f"Conservative Client:")
    print(f"   🧮 Mathematical Choice: {cons_aligned_doc['title'][:50]}...")
    print(f"   🤖 AI's Choice: {cons_naive_doc['title'][:50] if cons_naive_doc else 'Unknown'}...")
    print(f"   📊 Agreement: {'✅' if cons_aligned_doc['title'] == (cons_naive_doc['title'] if cons_naive_doc else '') else '❌'}")
    
    print(f"\nAggressive Client:")
    print(f"   🧮 Mathematical Choice: {agg_aligned_doc['title'][:50]}...")
    print(f"   🤖 AI's Choice: {agg_naive_doc['title'][:50] if agg_naive_doc else 'Unknown'}...")
    print(f"   📊 Agreement: {'✅' if agg_aligned_doc['title'] == (agg_naive_doc['title'] if agg_naive_doc else '') else '❌'}")
    
    # Quality Metrics Comparison
    print(f"\n📈 QUALITY METRICS ANALYSIS:")
    print("-"*50)
    
    if cons_naive_doc and agg_naive_doc:
        # Calculate average differences
        cred_diffs = [
            cons_aligned_doc['credibility'] - cons_naive_doc['credibility'],
            agg_aligned_doc['credibility'] - agg_naive_doc['credibility']
        ]
        
        risk_diffs = [
            cons_naive_doc['risk_level'] - cons_aligned_doc['risk_level'],
            agg_naive_doc['risk_level'] - agg_aligned_doc['risk_level']
        ]
        
        alignment_scores = [
            cons_aligned_doc.get('alignment_score', 0),
            agg_aligned_doc.get('alignment_score', 0)
        ]
        
        print(f"Average Credibility Advantage (Mathematical): {np.mean(cred_diffs):+.3f}")
        print(f"Average Risk Reduction (Mathematical): {np.mean(risk_diffs):+.3f}")
        print(f"Average Alignment Score: {np.mean(alignment_scores):.3f}")
        
        print(f"\n✅ Mathematical Approach Benefits:")
        if np.mean(cred_diffs) > 0:
            print(f"   • Consistently selects higher credibility sources")
        if np.mean(risk_diffs) > 0:
            print(f"   • Systematically reduces information risk exposure")
        print(f"   • Provides quantified alignment scoring for audit trails")
        print(f"   • Ensures client-specific personalization")
    
    # Client Personalization Analysis
    print(f"\n🎯 CLIENT PERSONALIZATION EFFECTIVENESS:")
    print("-"*50)
    
    cons_client_match = cons_aligned_doc.get('client_match', 0)
    agg_client_match = agg_aligned_doc.get('client_match', 0)
    
    print(f"Conservative Client Match Score: {cons_client_match:.3f}")
    print(f"Aggressive Client Match Score: {agg_client_match:.3f}")
    print(f"Personalization Differential: {abs(cons_client_match - agg_client_match):.3f}")
    
    if abs(cons_client_match - agg_client_match) > 0.1:
        print(f"✅ Strong personalization: Different documents selected for different client types")
    else:
        print(f"ℹ️ Moderate personalization: Similar documents appropriate for both clients")
    
    # System Performance Summary
    print(f"\n🚀 SYSTEM PERFORMANCE SUMMARY:")
    print("="*50)
    
    print(f"🧮 Mathematical Pre-filtering (Stage 1):")
    print(f"   ✅ Provides consistent, auditable document selection")
    print(f"   ✅ Integrates multiple quality factors: credibility, relevance, risk")
    print(f"   ✅ Enables systematic client personalization")
    print(f"   ✅ Creates transparent decision audit trail")
    
    print(f"\n🤖 AI Analysis (Stage 2):")
    if OPENAI_AVAILABLE:
        print(f"   ✅ Focuses on deep content analysis rather than document selection")
        print(f"   ✅ Operates on quality-assured, pre-filtered information")
        print(f"   ✅ Produces more confident, reliable recommendations")
        print(f"   ✅ Reduces risk of AI being misled by low-quality sources")
    else:
        print(f"   📋 Demonstrated through simulation (full analysis available with OpenAI API)")
        print(f"   💡 Mathematical pre-filtering provides foundation for AI analysis quality")
    
    print(f"\n🏆 OVERALL ASSESSMENT:")
    print(f"   📊 Two-stage approach successfully separates quality control from analysis")
    print(f"   ⚖️ Mathematical foundation ensures consistent standards across all queries")
    print(f"   🎯 Client-specific personalization works systematically")
    print(f"   🔍 Audit trail enables compliance and explainability requirements")
    
    if OPENAI_AVAILABLE:
        print(f"   ✅ Full AI integration demonstrates practical implementation")
    else:
        print(f"   💡 Framework ready for full AI integration with API access")

# Generate comprehensive summary
summarize_comparative_results(conservative_results, aggressive_results)

## Key Learning Outcomes - Part 2

You've successfully implemented the complete two-stage AI integration system:

### 1. **Two-Stage Architecture Mastery**
- ✅ **Stage 1 Implementation**: Mathematical document scoring and ranking
- ✅ **Stage 2 Integration**: OpenAI Responses API with specialized financial analysis
- ✅ **Quality Pre-filtering**: AI receives only mathematically validated information
- ✅ **Comparative Analysis**: Direct comparison of aligned vs naive approaches

### 2. **OpenAI Responses API Expertise**
- ✅ **Specialized Analyst Design**: Domain instructions + single stateless calls (no threads, no polling)
- ✅ **Dual-Mode Operation**: Both aligned and naive analysis capabilities
- ✅ **Robust Error Handling**: Graceful fallback to simulation when API unavailable
- ✅ **Response Processing**: Intelligent extraction of AI decision patterns

### 3. **AI Decision Quality Analysis**
- ✅ **Document Selection Tracking**: Which documents AI chooses independently
- ✅ **Quality Metric Comparison**: Credibility, risk, and relevance analysis
- ✅ **Consistency Assessment**: How mathematical guidance improves AI reliability
- ✅ **Client Personalization**: Systematic vs ad-hoc value alignment

## Technical Implementation Highlights

**Aligned Approach Process:**
1. Mathematical scoring ranks all documents
2. Top-ranked document sent to AI with quality metadata
3. AI focuses on analysis rather than document selection
4. Results include mathematical rationale for document choice

**Naive Approach Process:**
1. Raw documents presented to AI without quality indicators
2. AI must independently assess credibility and relevance
3. AI chooses document based on intuitive judgment
4. Analysis quality depends entirely on AI's document selection

## Demonstrated Benefits

*(Demonstrated on single comparative runs per client profile — Part 3 adds the repeated-trial evaluation needed to claim these benefits with statistical confidence.)*

**Consistency**: Mathematical approach provides same quality standards across all analyses

**Transparency**: Clear rationale for why specific documents were selected

**Risk Management**: Systematic assessment prevents low-quality information from reaching AI

**Personalization**: Client profiles mathematically integrated into document selection

## Next Steps

**Continue to Part 3** to see:
- Complete end-to-end demonstrations
- Advanced comparative experiments
- Real-world application scenarios
- Performance optimization techniques

**Advanced Experiments:**
1. Test with different document sets and quality distributions
2. Modify mathematical scoring parameters and observe AI behavior changes
3. Create edge case scenarios to stress-test the system
4. Implement additional client profile dimensions

The two-stage architecture you've built provides a robust foundation for **reliable, explainable, and personalized financial AI systems**!